# Bangla-LLM: End-to-End Training & Evaluation Notebook

This Jupyter Notebook contains the **complete pipeline** for fine-tuning, evaluating, and exporting the **BanglaSupport-LLM 7B** model on a GPU:

1. **Environment Setup**: Verifies CUDA availability and installs Unsloth QLoRA dependencies.
2. **Dataset Acquisition & Preprocessing**: Downloads `Bangla-Instruct` & `Aya Dataset`, performs NFC Unicode normalization, MinHash LSH deduplication, and structures intent data.
3. **QLoRA Fine-Tuning Execution**: Performs 4-bit NF4 QLoRA fine-tuning using Unsloth & `SFTTrainer` on Qwen2.5 / Qwen3 base models.
4. **Multi-Metric Evaluation**: Generates responses using the fine-tuned model and evaluates performance across **BLEU-4**, **ROUGE-L**, and **BERTScore**.
5. **Dual Weight Export**: Exports merged **Safetensors** for GPU deployment and 4-bit **GGUF** for fast CPU inference.

## Step 1: Install Dependencies

In [ ]:
%pip install --upgrade pip
%pip install --upgrade --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130
%pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install datasets transformers trl peft bitsandbytes sentencepiece protobuf
%pip install rouge-score nltk bert-score datasketch unicodedata2

In [ ]:
import torch

print("==========================================================")
print("PyTorch Environment Verification")
print("==========================================================")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Device: {device_name}")
    print(f"VRAM Capacity: {vram_gb:.2f} GB")
else:
    print("Warning: Running in CPU mode")

## Step 2: Dataset Pipeline 

In [ ]:
import os
import re
import json
import unicodedata
from datasets import load_dataset, Dataset
from datasketch import MinHash, MinHashLSH
from sklearn.model_selection import train_test_split

def classify_intent(text: str) -> str:
    return "general_faq"

def normalize_bangla_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text).strip()

print("1. Fetching instruction pairs from dataset...")
raw_samples = []

try:
    ds1 = load_dataset("md-nishat-008/Bangla-Instruct", split="train", streaming=True)
    for i, item in enumerate(ds1):
        if i >= 5000: break
        inst = normalize_bangla_text(item.get("instruction", ""))
        resp = normalize_bangla_text(item.get("response", ""))
        if inst and resp and len(inst) > 5:
            raw_samples.append({
                "instruction": inst,
                "context": "",
                "output": resp,
                "intent": classify_intent(inst)
            })
    print(f"   - Bangla-Instruct: {len(raw_samples)} pairs loaded")
except Exception as e:
    print(f"   - Bangla-Instruct notice: {e}")

try:
    ds2 = load_dataset("CohereForAI/aya_dataset", split="train", streaming=True)
    count = 0
    for item in ds2:
        if item.get("language_code") == "ben" or item.get("language") == "bengali":
            inst = normalize_bangla_text(item.get("inputs", ""))
            resp = normalize_bangla_text(item.get("targets", ""))
            if inst and resp and len(inst) > 5:
                raw_samples.append({
                    "instruction": inst,
                    "context": "",
                    "output": resp,
                    "intent": classify_intent(inst)
                })
                count += 1
                if count >= 1000: break
    print(f"   - Aya Dataset (Bengali): {count} pairs loaded")
except Exception as e:
    print(f"   - Aya Dataset notice: {e}")

print(f"2. Performing MinHash LSH deduplication on {len(raw_samples)} collected pairs...")
lsh = MinHashLSH(threshold=0.85, num_perm=128)
clean_data = []
for idx, sample in enumerate(raw_samples):
    text = sample["instruction"] + " " + sample["output"]
    m = MinHash(num_perm=128)
    for word in text.split():
        m.update(word.encode("utf-8"))
    result = lsh.query(m)
    if not result:
        lsh.insert(f"doc_{idx}", m)
        clean_data.append(sample)

print(f"   - Unique deduplicated pairs: {len(clean_data)}")

output_dir = "../Datasets"
os.makedirs(output_dir, exist_ok=True)
train_samples, val_samples = train_test_split(clean_data, test_size=0.1, random_state=42)

with open(os.path.join(output_dir, "train.jsonl"), "w", encoding="utf-8") as f:
    for item in train_samples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(os.path.join(output_dir, "val.jsonl"), "w", encoding="utf-8") as f:
    for item in val_samples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"3. Saved processed JSONL files in {output_dir}: {len(train_samples)} train / {len(val_samples)} val")

raw_dataset = load_dataset("json", data_files={
    "train": os.path.join(output_dir, "train.jsonl"),
    "validation": os.path.join(output_dir, "val.jsonl")
})
print("✓ Dataset Pipeline Completed Successfully!")

## Step 3: QLoRA Setup & Execution

In [ ]:
import os
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["UNSLOTH_ENABLE_GRADIENT_OFFLOADING"] = "0"

from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None
load_in_4bit = True

print("Loading 4-bit Base Model (Qwen2.5-7B-Instruct) with Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✓ QLoRA Model Initialized Successfully!")


In [ ]:
import os
import sys
import torch
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

max_seq_length = 512
output_dir = "checkpoints"
os.makedirs(output_dir, exist_ok=True)

class TargetLossCallback(TrainerCallback):
    def __init__(self, threshold=0.25, patience_steps=5, start_after_step=350):
        self.threshold = threshold
        self.patience_steps = patience_steps
        self.start_after_step = start_after_step
        self.low_loss_count = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step >= self.start_after_step and logs and "loss" in logs:
            current_loss = logs["loss"]
            if current_loss <= self.threshold:
                self.low_loss_count += 1
                print(f"\n[EarlyStopping] Step {state.global_step}: Loss ({current_loss:.4f}) <= {self.threshold} ({self.low_loss_count}/{self.patience_steps} logs)", flush=True)
                if self.low_loss_count >= self.patience_steps:
                    print(f"\n✓ Early Stopping Triggered after Step {self.start_after_step}! Optimal loss ({current_loss:.4f}) maintained for {self.patience_steps} logs.", flush=True)
                    control.should_training_stop = True
            else:
                self.low_loss_count = 0

def format_prompts(examples):
    texts = []
    for inst, out in zip(examples["instruction"], examples["output"]):
        prompt = f"<|im_start|>system\nতুমি একজন সহায়ক বাংলা ই-কমার্স গ্রাহক সেবা সহকারী।<|im_end|>\n<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        texts.append(prompt)
    return { "text" : texts }

train_files = []
for fname in ["train_ecom.jsonl", "train.jsonl"]:
    for prefix in ["../Datasets/", "Datasets/"]:
        candidate = prefix + fname
        if os.path.exists(candidate):
            train_files.append(candidate)
            break

if not train_files:
    raise RuntimeError("No training dataset files found in Datasets directory!")

loaded_datasets = [load_dataset("json", data_files={"train": tf})["train"] for tf in train_files]
train_data_to_format = concatenate_datasets(loaded_datasets)

formatted_dataset = train_data_to_format.map(format_prompts, batched = True)

batch_size = 2
grad_accum = 4
effective_batch_size = batch_size * grad_accum
total_epoch_steps = (len(formatted_dataset) + effective_batch_size - 1) // effective_batch_size

sft_config = SFTConfig(
    per_device_train_batch_size = batch_size,
    gradient_accumulation_steps = grad_accum,
    warmup_steps = 15,
    max_steps = total_epoch_steps,
    learning_rate = 3e-4,
    fp16 = not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    logging_steps = 1,
    save_strategy = "no",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = output_dir,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = sft_config,
    callbacks = [TargetLossCallback(threshold=0.25, patience_steps=5, start_after_step=350)],
)

if hasattr(trainer, "args") and hasattr(trainer.args, "__class__"):
    import trl.trainer.sft_config
    sys.modules["trl.trainer.sft_config"].SFTConfig = trainer.args.__class__
    if "trl" in sys.modules and hasattr(sys.modules["trl"], "SFTConfig"):
        sys.modules["trl"].SFTConfig = trainer.args.__class__

print("✓ SFTTrainer Configured!")
print(f"Max Steps: {total_epoch_steps} for {len(formatted_dataset)} samples.")
print("-----------------------------")
print("Starting QLoRA Fine-Tuning...", flush=True)
trainer_stats = trainer.train()
print("✓ Fine-Tuning Completed Successfully!", flush=True)

## Step 4: Multi-Metric Evaluation

In [ ]:
import os
import torch
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

FastLanguageModel.for_inference(model)
if hasattr(model, "generation_config") and model.generation_config:
    model.generation_config.max_length = None
    model.generation_config.max_new_tokens = 256

val_files = []
for fname in ["val_ecom.jsonl", "val.jsonl"]:
    for prefix in ["../Datasets/", "Datasets/"]:
        candidate = prefix + fname
        if os.path.exists(candidate):
            val_files.append(candidate)
            break

if not val_files:
    raise RuntimeError("No validation dataset files found in Datasets directory!")

loaded_val = [load_dataset("json", data_files={"val": vf})["val"] for vf in val_files]
from datasets import concatenate_datasets
val_dataset = concatenate_datasets(loaded_val)
eval_samples = list(val_dataset)[:50]

predictions = []
references = [x["output"] for x in eval_samples]

print(f"Generating answers from fine-tuned model for {len(eval_samples)} validation samples...", flush=True)
for idx, sample in enumerate(tqdm(eval_samples, desc="Evaluating")): 
    prompt = f"<|im_start|>system\nতুমি একজন সহায়ক বাংলা ই-কমার্স গ্রাহক সেবা সহকারী।<|im_end|>\n<|im_start|>user\n{sample['instruction']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = model.generate(**inputs, max_new_tokens=256, max_length=None, use_cache=True)
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    predictions.append(generated_text.strip())

smooth = SmoothingFunction().method1
bleu_scores = [sentence_bleu([ref.split()], pred.split(), smoothing_function=smooth) for pred, ref in zip(predictions, references)]
r_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
rouge_scores = [r_scorer.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(predictions, references)]

print("==========================================================")
print("--- Fine-Tuned Model Evaluation Results ---")
print("==========================================================")
print(f"Average BLEU-4 Score:  {sum(bleu_scores)/max(len(bleu_scores), 1):.4f}")
print(f"Average ROUGE-L Score: {sum(rouge_scores)/max(len(rouge_scores), 1):.4f}")


## Step 5: Save Models

In [ ]:
print("Saving Merged 16-bit Safetensors for GPU serving...")
model.save_pretrained_merged("models/BanglaLLM-7B", tokenizer, save_method = "merged_16bit")
print("✓ Saved to Research/models/BanglaLLM-7B/model.safetensors")

try:
    print("Saving 4-bit GGUF model for CPU serving...")
    model.save_pretrained_gguf("models", tokenizer, quantization_method = "q4_k_m")
    print("✓ Saved to Research/models/banglallm-7b-q4_k_m.gguf")
except Exception as e:
    print(f"GGUF Export note: {e}")